In [ ]:
# 23_mlp_lakes.ipynb
# Train intersection-min MLP on lakes-real-50k quadtree vectors.
# Same method as nb16 (intersection-min loss, GPU search, WJ rerank).
# Lakes vocab = 18,943-D  |  corpus = 40k  |  queries = 10k

# ── Configuration ────────────────────────────────────────────────────────────
run_cache    = True    # set False if CACHE_QT already exists
run_training = True
run_eval     = True
device_str   = "cuda:0"

ENC_DIR  = "/raid/ruban/hpmlproj/encoding/lakes-real50k0.022"
GT_DIR   = "/raid/ruban/hpmlproj/groundtruth/lakes-shapely-50K"
CACHE_QT = "/tmp/qt_lakes_50k.npy"
CACHE_GT = "/tmp/gt_lookup_lakes_50k.pkl"

LAKES_VOCAB  = 18943
POLY_COUNT   = 50_000
QUERY_START  = 40_000

batch_size          = 512
epochs              = 50
lr                  = 1e-3
weight_decay        = 1e-4
max_pos             = 30
intersection_margin = 0.01
candidate_ks        = [500, 1000]
rerank_batch_size   = 16
search_corpus_chunk = 4000

ckpt_path = "/tmp/best_compressor_intersection_min_lakes.pt"
out_path  = "/tmp/results_mlp_lakes.pkl"
seed = 42

In [ ]:
import gc
import glob
import os
import pickle
import random
import re
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

device = torch.device(device_str if torch.cuda.is_available() else "cpu")
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

print(f"device={device} | vocab={LAKES_VOCAB} | corpus={QUERY_START} | queries={POLY_COUNT-QUERY_START}")

In [ ]:
# ── Cache lakes quadtree vectors (real_*.txt, same format as parks) ───────────
def load_real_encodings(enc_dir, vocab_size, total):
    files = sorted(
        glob.glob(os.path.join(enc_dir, "real_*.txt")),
        key=lambda p: int(re.search(r'real_(\d+)\.txt', p).group(1))
    )
    print(f"Found {len(files)} encoding files")
    matrix = np.zeros((total, vocab_size), dtype=np.float32)
    poly_id = 0
    for fpath in tqdm(files, desc="Loading encodings"):
        start_id = int(re.search(r'real_(\d+)\.txt', fpath).group(1))
        rows = np.loadtxt(fpath, dtype=np.float32)
        if rows.ndim == 1:
            rows = rows.reshape(1, -1)
        n = min(len(rows), total - start_id)
        matrix[start_id:start_id + n] = rows[:n, :vocab_size]
    active = (matrix > 0).sum(axis=1).mean()
    print(f"Matrix {matrix.shape} | avg active cells: {active:.1f}")
    return matrix


# ── Cache GT (similarityMap_*.npy → dict {query_id: [neighbor_ids]}) ─────────
def load_gt_npy(gt_dir, query_start):
    """Lakes GT: each .npy file is (N, 500) int32 array of neighbor IDs.
    Filename encodes the query ID range: similarityMap_<start>-<end>.npy
    Value -1 = padding (no neighbor).
    """
    files = sorted(glob.glob(os.path.join(gt_dir, "similarityMap_*.npy")))
    gt = {}
    for fpath in tqdm(files, desc="Loading GT"):
        m = re.search(r'similarityMap_(\d+)-(\d+)\.npy', os.path.basename(fpath))
        start_qid = int(m.group(1))
        arr = np.load(fpath)  # shape (N, 500), dtype int32
        for i, row in enumerate(arr):
            qid = start_qid + i
            neighbors = [int(x) for x in row if x >= 0 and x < query_start]
            if neighbors:
                gt[qid] = neighbors
    print(f"GT loaded: {len(gt)} queries")
    return gt


if run_cache and not os.path.exists(CACHE_QT):
    print("Caching lakes quadtree vectors...")
    qt = load_real_encodings(ENC_DIR, LAKES_VOCAB, POLY_COUNT)
    np.save(CACHE_QT, qt)
    print(f"Saved {CACHE_QT}")
else:
    print(f"Loading cached: {CACHE_QT}")
    qt = np.load(CACHE_QT)

if run_cache and not os.path.exists(CACHE_GT):
    gt = load_gt_npy(GT_DIR, QUERY_START)
    with open(CACHE_GT, "wb") as f:
        pickle.dump(gt, f)
    print(f"Saved {CACHE_GT}")
else:
    with open(CACHE_GT, "rb") as f:
        gt = pickle.load(f)
    print(f"GT loaded from cache: {len(gt)} queries")

corpus_qt   = qt[:QUERY_START]
query_qt    = qt[QUERY_START:]
corpus_sums = corpus_qt.sum(axis=1)

print(f"qt={qt.shape} | corpus={corpus_qt.shape} | queries={query_qt.shape} | GT={len(gt)}")

In [ ]:
# ── Model + intersection-min loss (identical to nb16) ─────────────────────────
class QuadtreeCompressorMin(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096,   1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024,    512, bias=False), nn.BatchNorm1d(512),
        )

    def forward(self, x):
        out = self.net(x)
        out = F.relu(out)
        return out / out.sum(dim=1, keepdim=True).clamp(min=1e-10)


def intersection(a, b):
    return torch.minimum(a, b).sum(dim=-1)


def intersection_triplet_loss(anchors, positives, margin=0.01):
    i_pos    = intersection(anchors, positives)
    cross    = torch.min(anchors.unsqueeze(1), positives.unsqueeze(0)).sum(dim=2)
    cross.fill_diagonal_(-1e9)
    i_neg    = cross.max(dim=1).values
    loss     = F.relu(i_neg - i_pos + margin)
    violated = loss > 0
    if violated.sum() == 0:
        z = torch.tensor(0.0, device=anchors.device, requires_grad=True)
        return z, 0, i_pos.detach().mean(), i_neg.detach().mean()
    return loss[violated].mean(), int(violated.sum()), i_pos.detach().mean(), i_neg.detach().mean()


class AnchorPositiveDataset(Dataset):
    def __init__(self, qtree_vectors, gt_lookup, query_start, max_pos=30):
        self.vecs  = torch.tensor(qtree_vectors, dtype=torch.float32)
        self.pairs = []
        for qid, neighbors in gt_lookup.items():
            for nid in neighbors[:max_pos]:
                if nid < query_start:
                    self.pairs.append((qid, nid))
        random.shuffle(self.pairs)
        print(f"Anchor-positive pairs: {len(self.pairs):,}")

    def __len__(self):  return len(self.pairs)

    def __getitem__(self, idx):
        qid, pos_id = self.pairs[idx]
        return self.vecs[qid], self.vecs[pos_id]


print(f"Model defined — in_dim={LAKES_VOCAB}")

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────
def train(qt, gt, n_epochs):
    print("Training from scratch (random init).")
    model = QuadtreeCompressorMin(qt.shape[1]).to(device)

    dataset = AnchorPositiveDataset(qt, gt, QUERY_START, max_pos)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                         num_workers=4, pin_memory=True, drop_last=True)
    print(f"Steps/epoch: {len(loader)}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    best_loss = float("inf")
    for epoch in range(1, n_epochs + 1):
        model.train()
        total_loss, steps = 0.0, 0
        pbar = tqdm(loader, desc=f"Epoch {epoch:02d}/{n_epochs}", leave=False)
        for anchor, positive in pbar:
            anchor   = anchor.to(device, non_blocking=True)
            positive = positive.to(device, non_blocking=True)
            B = anchor.shape[0]
            out  = model(torch.cat([anchor, positive]))
            loss, n_viol, i_pos, i_neg = intersection_triplet_loss(
                out[:B], out[B:], intersection_margin)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += float(loss.detach().cpu())
            steps += 1
            pbar.set_postfix(loss=f"{float(loss):.4f}", viol=n_viol,
                             ipos=f"{float(i_pos):.3f}", ineg=f"{float(i_neg):.3f}")

        avg_loss = total_loss / max(steps, 1)
        scheduler.step()
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), ckpt_path)
        if epoch == 1 or epoch % 5 == 0 or epoch == n_epochs:
            print(f"Epoch {epoch:02d}/{n_epochs} | loss={avg_loss:.4f} | "
                  f"best={best_loss:.4f} | lr={scheduler.get_last_lr()[0]:.2e}")

    print(f"Saved {ckpt_path} (best_loss={best_loss:.4f})")
    return model


if run_training:
    model = train(qt, gt, epochs)
else:
    model = QuadtreeCompressorMin(LAKES_VOCAB).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    print(f"Loaded {ckpt_path}")
model.eval()

In [ ]:
# ── Eval helpers ──────────────────────────────────────────────────────────────
def generate_embeddings(model, data, batch_size=512):
    model.eval()
    chunks = []
    with torch.no_grad():
        for s in tqdm(range(0, len(data), batch_size), desc="Embedding"):
            batch = torch.tensor(data[s:s+batch_size], dtype=torch.float32, device=device)
            chunks.append(model(batch).cpu().numpy())
    embs = np.vstack(chunks)
    print(f"simplex sums: min={embs.sum(1).min():.4f} max={embs.sum(1).max():.4f}")
    return embs


def recall_at_k(gt_lookup, nbrs, query_start_id, k):
    total, count = 0.0, 0
    for i, ids in enumerate(nbrs):
        qid    = query_start_id + i
        gt_set = set(gt_lookup.get(qid, [])[:k])
        if not gt_set: continue
        total += len(gt_set & set(ids[:k])) / len(gt_set)
        count += 1
    return total / count if count else 0.0


def eval_recall(gt_lookup, nbrs, query_start_id, max_k):
    return {k: recall_at_k(gt_lookup, nbrs, query_start_id, k)
            for k in (10, 50, 100, 500) if k <= max_k}


@torch.no_grad()
def knn_intersection_gpu(query_embs, corpus_embs, k, corpus_chunk=4000):
    q     = torch.from_numpy(query_embs).to(device, dtype=torch.float32)
    c_all = torch.from_numpy(corpus_embs).to(device, dtype=torch.float32)
    n_q, n_c = q.shape[0], c_all.shape[0]
    top_ids    = np.zeros((n_q, k), dtype=np.int64)
    top_scores = np.full((n_q, k), -1.0, dtype=np.float32)

    for qs in tqdm(range(0, n_q, 64), desc="KNN intersection"):
        qe  = min(qs + 64, n_q)
        qb  = q[qs:qe]
        best_scores = torch.full((qb.shape[0], k), -1.0, device=device)
        best_ids    = torch.zeros((qb.shape[0], k), dtype=torch.long, device=device)
        for cs in range(0, n_c, corpus_chunk):
            ce       = min(cs + corpus_chunk, n_c)
            cb       = c_all[cs:ce]
            scores   = torch.min(qb[:, None, :], cb[None, :, :]).sum(dim=2)
            cand_ids = torch.arange(cs, ce, device=device).expand(qb.shape[0], -1)
            merged_s = torch.cat([best_scores, scores], dim=1)
            merged_i = torch.cat([best_ids, cand_ids], dim=1)
            new_s, order = torch.topk(merged_s, k=min(k, merged_s.shape[1]), dim=1)
            best_ids    = torch.gather(merged_i, 1, order)
            best_scores = new_s
        top_ids[qs:qe]    = best_ids.cpu().numpy()
        top_scores[qs:qe] = best_scores.cpu().numpy()

    return [(top_ids[i].tolist(), top_scores[i].tolist()) for i in range(n_q)]


def rerank_wj_gpu(query_qt, nbrs_ids, corpus_qt, corpus_sums, batch_size=16):
    corpus_t      = torch.from_numpy(corpus_qt).to(device, dtype=torch.float32)
    corpus_sums_t = torch.from_numpy(corpus_sums).to(device, dtype=torch.float32)
    reranked = []
    for s in tqdm(range(0, len(nbrs_ids), batch_size), desc="WJ rerank"):
        batch  = nbrs_ids[s:s+batch_size]
        groups = {}
        for offset, ids in enumerate(batch):
            ids_arr = np.asarray(ids, dtype=np.int64)
            groups.setdefault(len(ids_arr), []).append((s + offset, ids_arr))
        for cand_len, items in groups.items():
            if cand_len == 0:
                for _ in items: reranked.append([])
                continue
            ids_np   = np.stack([ids for _, ids in items])
            query_np = np.stack([query_qt[abs_i] for abs_i, _ in items])
            ids_t    = torch.from_numpy(ids_np).to(device)
            q_t      = torch.from_numpy(query_np).to(device, dtype=torch.float32)
            c_t      = corpus_t[ids_t]
            mins     = torch.minimum(q_t[:, None, :], c_t).sum(dim=2)
            maxs     = q_t.sum(dim=1, keepdim=True) + corpus_sums_t[ids_t] - mins
            order    = torch.argsort(mins / maxs.clamp_min(1e-10),
                                     dim=1, descending=True).cpu().numpy()
            for row, (_, ids) in zip(order, items):
                reranked.append(ids[row].tolist())
    del corpus_t, corpus_sums_t
    torch.cuda.empty_cache()
    return reranked


print("Eval functions defined.")

In [ ]:
# ── Run eval ──────────────────────────────────────────────────────────────────
if run_eval:
    if Path(ckpt_path).exists():
        model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    model.eval()

    print("\n" + "="*72)
    print("MLP INTERSECTION-MIN — LAKES 50k")
    print("="*72)

    embs        = generate_embeddings(model, qt)
    corpus_embs = embs[:QUERY_START]
    query_embs  = embs[QUERY_START:]

    results = {}
    max_k   = max(max(candidate_ks), 500)

    # Stage 1
    t0 = time.time()
    nbrs     = knn_intersection_gpu(query_embs, corpus_embs, k=max_k,
                                    corpus_chunk=search_corpus_chunk)
    qps_s1   = len(query_embs) / (time.time() - t0)
    ids_only = [ids for ids, _ in nbrs]

    print(f"\n--- Stage 1: top-{max_k} by INTERSECTION on 512-D ---")
    rec = eval_recall(gt, ids_only, QUERY_START, max_k)
    results["intersection_no_rerank"] = {**rec, "qps": qps_s1}
    for k, r in rec.items(): print(f"  R@{k:<4} = {r:.4f}")
    print(f"  QPS ≈ {qps_s1:.1f}")

    # Stage 2
    for k in candidate_ks:
        print(f"\n--- Stage 2: top-{k} candidates + raw WJ rerank ---")
        cand_ids = [ids[:k] for ids, _ in nbrs]
        t0   = time.time()
        rr_ids = rerank_wj_gpu(query_qt, cand_ids, corpus_qt, corpus_sums, rerank_batch_size)
        qps  = len(query_embs) / (time.time() - t0)
        rec_rr = eval_recall(gt, rr_ids, QUERY_START, k)
        results[f"k{k}_wj_rerank"] = {**rec_rr, "qps": qps, "k": k}
        for rk, rv in rec_rr.items(): print(f"  R@{rk:<4} = {rv:.4f}")
        print(f"  QPS ≈ {qps:.1f}")

    payload = {
        "lakes_50k": results,
        "_meta": {
            "method":  "MLP_intersection_min",
            "dataset": "lakes_50k",
            "in_dim":  LAKES_VOCAB,
            "out_dim": 512,
            "loss":    "intersection_triplet_sum_min",
            "init":    "scratch",
            "ckpt":    ckpt_path,
            "time":    __import__('time').strftime("%Y-%m-%d %H:%M:%S"),
        }
    }
    with open(out_path, "wb") as f:
        pickle.dump(payload, f)
    print(f"\nSaved {out_path}")

    # ── Summary ───────────────────────────────────────────────────────────────
    print("\n" + "="*72)
    print("COMPARISON: Parks vs Lakes — same method, different dataset")
    print("="*72)
    print(f"{'Method':<35} {'Stage-1 R@10':>13} {'K=500 R@10':>11} {'K=1000 R@10':>12}")
    print("-"*72)
    print(f"{'Parks nb16 (18,499-D → 512-D)':<35} {'0.6726':>13} {'0.9958':>11} {'0.9961':>12}")
    s1  = results['intersection_no_rerank'].get(10, float('nan'))
    k5  = results.get('k500_wj_rerank',  {}).get(10, float('nan'))
    k10 = results.get('k1000_wj_rerank', {}).get(10, float('nan'))
    print(f"{'Lakes  (18,943-D → 512-D)':<35} {s1:>13.4f} {k5:>11.4f} {k10:>12.4f}")

gc.collect()
torch.cuda.empty_cache()